In [1]:
import poolparty as pp 
pp.init()
pp.add_highlight(which='tags', style='gray')
pp.add_highlight(region='cre', style='lightskyblue')
pp.add_highlight(which='lower gap', style='bold yellow')


In [2]:
# 1. Create template Pool
template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')
template_pool.print_library()

pool[0]: seq_length=51, num_states=1
state  seq
    0  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA



Pool(id=0, name='pool[0]', op='op[0]:from_seq', num_states=1)

In [3]:
# 2. Randomly mutate cre region
mutagenized_pool = pp.mutagenize(template_pool, 
                                 region='cre', 
                                 mutation_rate=0.1, 
                                 mode='hybrid', 
                                 num_hybrid_states=5, 
                                 seq_name_prefix='mut_')
mutagenized_pool.print_library()


pool[1]: seq_length=51, num_states=5
state  name   seq
    0  mut_0  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc/>AGATCGGA
    1  mut_1  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc/>AGATCGGA
    2  mut_2  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc/>AGATCGGA
    3  mut_3  TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc/>AGATCGGA
    4  mut_4  TCCCGACT<cre>GGgAAGCGGGCtGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA



Pool(id=1, name='pool[1]', op='op[1]:mutagenize', num_states=5)

In [14]:
# 3. Repeat each mutant 3 times
# 4. Insert random barcodes at <bc/>
mutagenized_bc_pool = mutagenized_pool.repeat_states(3, seq_name_prefix='v', op_iter_order=-1).insert_kmers('bc', length=5)
mutagenized_bc_pool.print_library()

pool[13]: seq_length=5, num_states=15
state  name      seq
    0  mut_0.v0  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_0.v2  TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v0  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_1.v1  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_1.v2  TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_2.v0  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_2.v1  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_2.v2  TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_3.v0  TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>gggcg</bc>AGATCGGA
 

Pool(id=13, name='pool[13]', op='op[13]:get_kmers', num_states=15)

In [17]:
pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='hybrid', 
                                        num_hybrid_states=5, 
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1, 
                        seq_name_prefix='site_').named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential', 
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool])\
    .repeat_states(2, seq_name_prefix='v', op_iter_order=-2)\
    .insert_kmers('bc', length=5)\
    .named('combo_pool')\

combo_pool.print_library()

combo_pool: seq_length=5, num_states=40
state  name             seq
    0  mut_0.v0         TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  mut_0.v1         TCCCGACT<cre>GGAAAGaaGGCAGgGAGCACAaAaGAAA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  mut_1.v0         TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>gtggt</bc>AGATCGGA
    3  mut_1.v1         TCCCGACT<cre>GGAAAGgGGGCAGTGAGCACgCAGGAtc</cre>ATTACGG<bc>ggggt</bc>AGATCGGA
    4  mut_2.v0         TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    5  mut_2.v1         TCCCGACT<cre>GGAAAGCaGGCAGTtAGCACACAaaAcA</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    6  mut_3.v0         TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>taata</bc>AGATCGGA
    7  mut_3.v1         TCCCGACT<cre>GGAAtGCGGGCAGTGAaatCACAGGgAA</cre>ATTACGG<bc>gaccc</bc>AGATCGGA
    8  mut_4.v0         TCCCGACT<cre>GGgAAGCGGGCtGTGAGCACACAGGAAA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    9  mut_4.v1        

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_states=40)

In [6]:
df = combo_pool.clear_annotation()\
    .generate_library(report_design_cards=False)
df.to_excel('~/Desktop/library.xlsx', index=False)

In [7]:
combo_pool.print_dag()

combo_pool (pool, n=40)
└── op[10]:get_kmers [mode=random, n=1]
    └── pool[9] (pool, n=40)
        └── op[9]:repeat [mode=sequential, n=2]
            └── pool[8] (pool, n=20)
                └── op[8]:stack [mode=fixed, n=1]
                    ├── mutated_pool (pool, n=5)
                    │   └── op[1]:mutagenize [mode=hybrid, n=5]
                    │       └── template_pool (pool, n=1)
                    │           └── op[0]:from_seq [mode=fixed, n=1]
                    ├── deletion_pool (pool, n=5)
                    │   └── op[3]:deletion_scan(from_seq) [mode=fixed, n=1]
                    │       └── pool[2] (pool, n=5)
                    │           └── op[2]:deletion_scan(marker_scan) [mode=sequential, n=5]
                    │               └── template_pool (pool, n=1)
                    │                   └── op[0]:from_seq [mode=fixed, n=1]
                    └── insertion_pool (pool, n=10)
                        └── op[7]:insertion_scan(replace_marker_con

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_states=40)

In [18]:
combo_pool.counter.print_dag()

combo_pool.state (counter, io=-2, n=40)
└── [op=Product]
    ├── op[9]:repeat.state (counter, io=-2, n=2)
    ├── pool[8].state (counter, io=-1, n=20)
    │   └── [op=Stack]
    │       ├── mutated_pool.state (counter, io=0, n=5)
    │       │   └── [op=Product]
    │       │       ├── op[1]:mutagenize.state (counter, io=0, n=5)
    │       │       └── template_pool.state (counter, io=0, n=1)
    │       │           └── [op=Passthrough]
    │       │               └── op[0]:from_seq.state (counter, io=0, n=1)
    │       ├── deletion_pool.state (counter, io=0, n=5)
    │       │   └── [op=Product]
    │       │       ├── op[3]:deletion_scan(from_seq).state (counter, io=0, n=1)
    │       │       ├── op[2]:deletion_scan(marker_scan).state (counter, io=0, n=5)
    │       │       └── template_pool.state (counter, io=0, n=1)
    │       │           └── [op=Passthrough]
    │       │               └── op[0]:from_seq.state (counter, io=0, n=1)
    │       └── insertion_pool.state (counter,

In [9]:
counter_df = combo_pool.counter.get_iteration_df()
counter_df.to_excel('~/Desktop/counter_states.xlsx')

In [10]:
import statecounter as sc
A = sc.Counter(3, name='A')
B = sc.Counter(4, name='B')
C = sc.product([A,B], name='C')
C.get_iteration_df().T


C,0,1,2,3,4,5,6,7,8,9,10,11
A,0,1,2,0,1,2,0,1,2,0,1,2
B,0,0,0,1,1,1,2,2,2,3,3,3


In [11]:
import statecounter as sc
A = sc.Counter(3, name='A')
B = sc.Counter(4, name='B')
C = sc.stack([A,B], name='C')
C.get_iteration_df().T

C,0,1,2,3,4,5,6
A,0,1,2,None,None,None,None
B,None,None,None,0,1,2,3


In [12]:
import statecounter as sc
A = sc.Counter(2,name='A')
C = sc.repeat(A,3,name='C')
C.get_iteration_df().T

C,0,1,2,3,4,5
A,0,1,0,1,0,1


In [13]:
import statecounter as sc
A = sc.Counter(12,name='A')
C = sc.slice(A,None,None,3,name='C')
C.get_iteration_df().T

C,0,1,2,3
A,0,3,6,9
